<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_97_normalizing_flows_likelihood_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🌊 **Module 97 — Normalizing Flows + Likelihood Models** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 14 · Theory & Responsibility (UDL deep-dives)


# Module 97 — Normalizing Flows + Likelihood Models

> M85 covered GAN/AE/VAE; M86 covered diffusion. **Normalizing flows are the third generative-model family** — bijective neural networks that compute *exact* likelihoods via the change-of-variables formula. Built around UDL Chap16's three notebooks (**1D Normalizing Flows · Autoregressive Flows · Contraction Mappings**) and modernized to 2026: **NICE → RealNVP → Glow → MAF → IAF → Neural ODE → FFJORD → Score-Based / Continuous Flows**. By the end you'll know when to pick a flow over diffusion or VAE — exact likelihood, fast inference, density estimation, anomaly detection, RL distribution modeling, normalizing-flow MCMC.

### What you'll cover
1. The **change of variables** formula — why flows exist at all
2. Building blocks — **bijection + Jacobian-determinant**
3. **NICE** (Dinh 2014) — coupling layers with triangular Jacobian
4. **RealNVP** (Dinh 2017) — affine coupling, the modern coupling-flow template
5. **Glow** (Kingma 2018) — invertible 1×1 conv, ActNorm, image-quality flow
6. **Masked Autoregressive Flows** (MAF) vs **Inverse Autoregressive Flows** (IAF)
7. **Continuous Normalizing Flows / Neural ODE / FFJORD** (Chen 2018, Grathwohl 2019)
8. **Score-based / Diffusion** as a flow — the unification with M86
9. **When to use a flow** — vs GAN / VAE / diffusion / autoregressive LM
10. The **2026 stack** — nflows, FrEIA, normflows, JAX-Flows, Bayesian inference applications


## 1 · Change of variables — the equation that makes flows exist

Suppose `z ~ p_Z(z)` is a simple distribution (Gaussian) and `x = f(z)` for an **invertible** `f`. The change-of-variables formula gives the density of `x`:

```
p_X(x) = p_Z(f⁻¹(x)) · | det J_{f⁻¹}(x) |
```

Take the log (training is done in log-space):

```
log p_X(x) = log p_Z(f⁻¹(x))  +  log | det J_{f⁻¹}(x) |
```

**Train** by maximizing `log p_X(x)` over a dataset. **Sample** by drawing `z ~ N(0, I)` and computing `x = f(z)`. Exact likelihood. Single forward pass to sample. No reverse process. **No reconstruction loss, no adversarial loss, no denoising loss.**

The three engineering challenges:
1. **f must be invertible** — a free choice with no structural constraints would give terrible Jacobians or unstable inverses
2. **det J must be tractable** — for a 1024×1024 image, `J` is a `~10⁶ × 10⁶` matrix; computing its determinant takes `O(N³)`
3. **f must be expressive enough** to map a simple Gaussian to (say) the ImageNet distribution

Every flow paper since NICE 2014 is a different way to satisfy all three at once.

> 📐 **The intellectual lineage.** Flows are *exact* — likelihood is computable, no ELBO bound (VAE) and no `O(T)` Markov chain (diffusion). VAE was the 2013 "let's tractably model a generative process" approach; flows are 2014's "let's make the model itself the bijection so we don't need an approximate posterior."


## 2 · The two ingredients — bijection + tractable Jacobian

You compose K bijections `x = f_K ∘ ... ∘ f_1(z)`. The Jacobian determinant is the **product** of per-layer Jacobian determinants:

```
log | det J_f(z) | = Σ_k log | det J_{f_k}(z_{k-1}) |
```

So you only need each `f_k` to have a tractable Jacobian. Three structural tricks dominate:

| Trick | Layer is | det J via |
|---|---|---|
| **Triangular** | mask half of `x`, transform the other half conditional on it | product of diagonal entries |
| **Autoregressive** | output `i` depends on inputs `1..i` only | product of diagonal entries (lower-triangular) |
| **1×1 convolution** | reshape image, multiply by learned `W ∈ ℝ^{C×C}` | `det(W)` is C-dim, cheap |
| **Free-form / continuous** | parametrize ODE | trace of velocity-field Jacobian (Hutchinson estimator) |

Coupling layers (next) implement the triangular trick. Glow adds the 1×1 conv. MAF/IAF do the autoregressive variant. FFJORD does continuous.


## 3 · NICE — coupling layers with triangular Jacobian

**NICE** (Non-linear Independent Components Estimation, Dinh, Krueger, Bengio 2014) was the first practical flow. The **additive coupling layer**:

```
Split  x = [x_1, x_2]                # half the dimensions in each
y_1 = x_1                            # identity on the first half
y_2 = x_2 + m_θ(x_1)                 # additive shift; m is any NN

Inverse:
x_1 = y_1
x_2 = y_2 - m_θ(y_1)                 # exactly inverse, even though m is nonlinear

Jacobian:  [ I       0 ]            # determinant = 1
           [ ∂m/∂x_1  I ]
```

The Jacobian determinant is **identically 1** — the change-of-variables term contributes nothing. Training reduces to `log p_Z(f⁻¹(x))`. Limitation: no learned scale → restricted expressiveness. RealNVP fixed it.

> 💡 **The coupling trick.** This is the central engineering idea in flows. By making half of the output an exact copy and the other half a deterministic function of the first half, we get a network that's bijective *by construction* and has a triangular Jacobian. Cost: every layer touches only half the variables, so you need many layers (and many shuffle / channel-permutation layers between them).


## 4 · RealNVP — affine coupling, the modern template

**RealNVP** (Dinh, Sohl-Dickstein, Bengio 2017) generalizes NICE with **affine** coupling:

```
y_1 = x_1
y_2 = x_2 · exp(s_θ(x_1)) + t_θ(x_1)        # element-wise scale + shift

Inverse:
x_1 = y_1
x_2 = (y_2 - t_θ(y_1)) · exp(-s_θ(y_1))

log | det J | = Σ_i s_θ(x_1)_i              # the scale's log adds to the loss
```

Now the Jacobian determinant is non-trivial (the scale gives non-unit volume change) and the model can learn density. RealNVP introduced two other still-used ideas:

1. **Checkerboard + channel masks** — alternate between spatial-checkerboard and channel-wise splits to mix dimensions
2. **Multi-scale architecture** — at each level, send half the channels to the output (Gaussian) and continue processing the rest at lower resolution, like a U-Net but invertible

Trained on CelebA, ImageNet 32×32. Competitive samples for 2017; the architecture (coupling + multi-scale) survives in 2026 flow research.


In [ ]:
import torch, torch.nn as nn

class AffineCoupling(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim // 2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, dim),      # outputs s, t each dim/2-D
        )
    def forward(self, x):                              # returns y, log|det J|
        x1, x2 = x.chunk(2, dim=-1)
        st = self.net(x1); s, t = st.chunk(2, dim=-1)
        s = torch.tanh(s)                              # keep |s| bounded
        y1 = x1
        y2 = x2 * s.exp() + t
        logdet = s.sum(dim=-1)                          # Σ log(exp(s)) = Σ s
        return torch.cat([y1, y2], dim=-1), logdet
    def inverse(self, y):
        y1, y2 = y.chunk(2, dim=-1)
        st = self.net(y1); s, t = st.chunk(2, dim=-1)
        s = torch.tanh(s)
        x1 = y1
        x2 = (y2 - t) * (-s).exp()
        return torch.cat([x1, x2], dim=-1)


## 5 · Glow — image-quality flow

**Glow** (Kingma & Dhariwal NeurIPS 2018) added two improvements that made flows produce **CelebA-HQ-quality images at 256×256**:

| Component | What it does |
|---|---|
| **ActNorm** | Per-channel affine; like BN but data-dependent init (no batch dependence at test time) |
| **Invertible 1×1 conv** | Replace fixed channel permutation with a learned `W ∈ ℝ^{C×C}`; `det(W)` is cheap (C is small) |
| **Affine coupling** (RealNVP) | As before |

Stacked many times across a multi-scale pyramid. The 1×1 conv is the key — it makes channel-mixing learnable rather than fixed.

**Speed/quality trade-off.** A Glow model trained on CelebA-HQ has ~200M parameters and samples in ~one second on a GPU. That's *faster* than diffusion (M86) and **exact-likelihood** to boot. But quality stayed below the GAN/diffusion frontier and the parameter count is large per pixel.

> 🧪 **Why flows didn't win the image race.** RealNVP/Glow have to maintain *full-dimensional* latent state at every layer (no bottleneck like VAE), and the affine-coupling restriction limits per-layer expressiveness. To match a diffusion model on FID, a flow needs ~10× the parameters. The remaining flow community pivoted to (a) lower-dimensional problems (density estimation, anomaly detection, posterior inference) and (b) **continuous flows** (next) where the architecture is more flexible.


## 6 · Autoregressive flows — MAF vs IAF

A different way to get a triangular Jacobian: **make the i-th output depend on inputs 1..i** (lower-triangular).

**MAF (Masked Autoregressive Flow, Papamakarios 2017):**
```
y_i = f(x_i ; μ_θ(x_{1..i-1}), σ_θ(x_{1..i-1}))
```

The Jacobian is lower-triangular → `det J = Π σ_θ(x_{1..i-1})`. **Fast density evaluation** (`log p` in one pass via MADE/PixelCNN-style masks), **slow sampling** (need to do D sequential steps).

**IAF (Inverse Autoregressive Flow, Kingma 2016):**
```
y_i = f(z_i ; μ_θ(z_{1..i-1}), σ_θ(z_{1..i-1}))   # autoregressive in z, not x
```

**Fast sampling** (one parallel pass), **slow density** (need D sequential steps).

| | MAF | IAF |
|---|---|---|
| Density `p(x)` | parallel | sequential |
| Sample | sequential | parallel |
| When | density estimation, anomaly | VAE posterior (Kingma 2016 IAF-VAE) |

**Neural Spline Flows** (Durkan 2019) replaced the affine `f` with a monotonic rational-quadratic spline — much more expressive per layer. Modern flows almost always use spline transforms in the coupling or autoregressive layers.

> 🔄 **MAF and IAF are inverses of each other** — same architecture, but train/sample roles swapped. This duality is the structural insight of the autoregressive-flow line.


## 7 · Continuous Normalizing Flows — Neural ODEs and FFJORD

**Neural ODE** (Chen, Rubanova, Bettencourt, Duvenaud NeurIPS 2018, best paper):
```
dz/dt = f_θ(z(t), t)
```

A "layer" is a continuous-time ODE step. Forward = ODE integration; backward = adjoint method (also an ODE). The instantaneous change-of-variables formula gives:

```
∂ log p(z(t))/∂t = -Tr( ∂f_θ/∂z )
```

**FFJORD** (Grathwohl, Chen, Bettencourt, Sutskever, Duvenaud ICLR 2019) made this practical by replacing the exact trace with **Hutchinson's stochastic estimator** `E[v^T (∂f/∂z) v]` for random `v ~ N(0,I)`. Now the trace is a single Jacobian-vector product → tractable for any `f`.

Continuous flows give:
- **Free-form `f`** — no triangular constraint, no coupling — any neural network as the vector field
- **Exact likelihood** — trace evaluation is cheap with Hutchinson
- **Reversibility** — integrate the ODE backwards
- **Cost** — ~10-50 ODE solver steps per forward; expensive in 2018, more manageable in 2026

**Flow Matching** (Lipman, Chen, Ben-Hamu, Nickel, Le 2023) trains a continuous flow without ever solving the ODE during training — just match the velocity field to an analytical interpolation. This is the recipe behind **FLUX.1** (M86) and **Stable Diffusion 3 / 3.5**. *Diffusion and flows have become the same family.*

> 🌀 **The unification.** Score-based diffusion (M21, M86) is a continuous flow whose velocity field is `-∇ log p_t(x)`. Flow-matching trains the velocity field directly. **CNF + score-based + flow-matching = one continuous-time framework**, parameterized by what loss you use to fit the velocity field.


## 8 · Score-based & diffusion as a flow

**Score-based models** (Song & Ermon 2019, 2020) learn the score `s_θ(x, t) ≈ ∇_x log p_t(x)` and sample via Langevin dynamics or ODE integration. The continuous-time SDE view (Song 2021):

```
Forward SDE:   dx = f(x, t) dt + g(t) dW
Reverse SDE:   dx = [f(x, t) - g(t)² s_θ(x, t)] dt + g(t) dW̃
Probability flow ODE:  dx = [f(x, t) - ½ g(t)² s_θ(x, t)] dt
```

The probability-flow ODE is a **continuous normalizing flow** — same equation as Section 7, different way of training the velocity field. It produces samples deterministically (no noise in reverse) and lets you compute exact likelihoods.

This is the bridge between the 2016-2019 flow literature and the 2020-2026 diffusion explosion. From a 2026 vantage point:

| Model family | What they do | When used |
|---|---|---|
| **Discrete flows** (NICE/RealNVP/Glow) | Stack bijections; exact likelihood | Density estimation, anomaly detection, RL |
| **Autoregressive flows** (MAF, IAF) | Triangular Jacobian via autoregression | Posterior inference, VAE flows |
| **Continuous flows** (Neural ODE, FFJORD) | ODE-based velocity field; Hutchinson trace | Modeling, MCMC, physics, mol gen |
| **Score-based / Diffusion** | Match score `∇ log p_t`; SDE or ODE | Images, audio, video — the 2026 default |
| **Flow Matching** | Match velocity field to analytical interpolation | FLUX, SD3+, video diffusion |

The same machinery, four different ways to learn the same underlying object.


## 9 · When to use a flow — and when not to

Flows shine when you need things diffusion / GANs / VAEs can't give you:

| Need | Flow wins because |
|---|---|
| **Exact log-likelihood** | No ELBO bound, no Markov-chain integration |
| **Density estimation** | Direct `log p(x)` for any `x` |
| **Anomaly / OOD detection** | Lowest `log p(x)` = most anomalous |
| **MCMC posterior sampling** | Reparameterize a complex posterior as a flow over Gaussian noise |
| **Reinforcement learning** | Differentiable, exact policies and value distributions (Soft Actor-Critic-Flow) |
| **Variational inference** | IAF in a VAE gives expressive posteriors |
| **Audio / molecules** | WaveNet → WaveGlow; flows for molecular conformation |
| **Optimal transport** | Continuous flows ≈ Schrödinger bridge ≈ rectified flow |

When *not* to use a flow:
- **Image generation at frontier quality** → diffusion / flow-matching (M86)
- **Text generation** → autoregressive Transformer (M19/M20)
- **Latent representation learning** for downstream tasks → VAE or contrastive (M85)
- **GPU memory constrained** → flows need full-dim state per layer

> 🎯 **The 2026 take.** Flows lost the image-quality race but won the **rigor** race. Anywhere a tractable likelihood matters — physics, finance, scientific ML, Bayesian deep learning, reinforcement learning — flows are still the default. And **flow matching** brought the architecture back to the forefront of the diffusion-image-generation world.


## 10 · The 2026 stack

| Library | Maintainer | Use |
|---|---|---|
| **nflows** | Durkan / Edinburgh | Reference PyTorch library; spline flows, MAF/IAF, RealNVP, Glow |
| **normflows** | Stimper | Active PyTorch lib; conditional flows, neural ODE, Resampled-RealNVP |
| **FrEIA** | Heidelberg | Framework for Easily Invertible Architectures; production-tuned |
| **Distrax / TFP** | DeepMind / Google | JAX flows + bijectors (TensorFlow Probability inherits) |
| **Pyro / NumPyro** | Uber / Linux Foundation | Bayesian inference with flows as posteriors |
| **TorchDiffEq / DiffEqFlux.jl** | Chen / SciML | Neural ODE solvers |
| **FlowMatching (Meta)** | Lipman et al. | Reference flow-matching implementation |

**Code in 15 lines** to fit a 1D normalizing flow:
```python
import torch, normflows as nf
base = nf.distributions.DiagGaussian(2)
layers = [nf.flows.AffineCouplingBlock(nf.nets.MLP([1, 64, 64, 2]))
          for _ in range(8)]
model = nf.NormalizingFlow(base, layers)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(2000):
    x = sample_target()                         # your 2D data
    loss = -model.log_prob(x).mean()
    opt.zero_grad(); loss.backward(); opt.step()
samples = model.sample(1024)[0]                 # done
```

That's the entire pipeline for density estimation + sampling + likelihood eval — a few dozen lines for what would have been a research paper in 2016.

> 🎓 **The takeaway**. Flows are the *third* (and oldest of the modern three) generative-model family — the rigorous one. They're niche in 2026 image gen, but central to scientific ML, Bayesian inference, RL, and the continuous-time framework that now subsumes diffusion. Knowing them rounds out the **GAN + VAE + flow + diffusion + autoregressive** generative-model basis.

## ✅ Recap

- **Change-of-variables**: `log p_X(x) = log p_Z(f⁻¹(x)) + log|det J|`. Exact likelihood, single-pass sampling.
- **Coupling layers** (NICE/RealNVP/Glow) → triangular Jacobian via "half identity, half conditional transform."
- **Autoregressive flows** (MAF/IAF) → lower-triangular Jacobian via masked autoregression; MAF and IAF are mutual inverses.
- **Spline coupling** (Neural Spline Flows) → more expressive per layer; modern default.
- **Continuous flows** (Neural ODE/FFJORD) → free-form velocity field + Hutchinson trace estimator.
- **Diffusion + score-based + flow-matching = continuous flows** with different training objectives — the 2026 unified framework.
- **Use flows when** you need exact likelihood: density estimation, anomaly, MCMC, Bayesian inference, RL.
- **2026 stack**: nflows · normflows · FrEIA · Distrax · NumPyro · TorchDiffEq · FlowMatching.

Next: **M98 — Generalization Theory + Bayesian DL**.
